# 06 — Demo interactiva

Este notebook ofrece dos formas complementarias de ver el clasificador en acción:

- **Modo A — Dibujo libre** (secciones A–E): el alumno elige un compuesto, lo dibuja en un canvas y el modelo intenta reconocerlo. La interacción más vistosa pero también la más exigente, porque el modelo no se entrenó con dibujos a mano humanos sino con renders de RDKit. Por defecto cargamos el modelo `best_model_handwritten.pt` si existe (entrenado con `HANDWRITTEN_TRAIN_TRANSFORM`, ver `scripts/retrain_handwritten.py`); este es más robusto a dibujos a mano que el `best_model.pt` original del notebook 04, aunque sigue lejos del 99% del test del dataset.
- **Modo B — Ver al modelo trabajar** (sección F): toma una imagen aleatoria del split de test, la pasa por el modelo y muestra la predicción + top-5. Aquí el modelo trabaja sobre el dominio en el que fue entrenado, así que la accuracy es ~99,5%.

### Por qué dos modos

El modo A demuestra el caso de uso real (un alumno dibujando), pero está limitado por el cambio de dominio entre RDKit y mano humana. El modo B demuestra el rendimiento real del modelo sobre su dominio de entrenamiento. Juntos pintan la fotografía completa: el modelo es muy bueno en lo suyo (99,5% en test), pero no generaliza por sí solo a una distribución muy distinta (dibujos a mano libre) sin reentrenamiento específico.

Antes de cualquier interacción cargamos `saved_models/best_model_handwritten.pt` (o `best_model.pt` como fallback) junto con su `*_config.json`. Si no existen, mostramos un aviso explícito; basta con ejecutar el notebook 04 (o el script de reentrenamiento) y recargar este cuaderno.

In [1]:
import sys, json, random
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch, numpy as np
torch.manual_seed(42); np.random.seed(42); random.seed(42)

MODELS_DIR = ROOT / 'saved_models'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


In [2]:
from src import PretrainedModel, VAL_TRANSFORM
from data.compounds import COMPOUNDS, TAXONOMY, get_compounds

# Buscamos primero el modelo entrenado con augmentacion agresiva
# (best_model_handwritten.pt) por ser mas robusto a dibujos a mano.
# Si no existe, caemos al modelo original (best_model.pt) del notebook 04.
candidates = [
    (MODELS_DIR / 'best_model_handwritten_config.json',
     MODELS_DIR / 'best_model_handwritten.pt',
     'handwritten-aug (robusto a dibujos a mano)'),
    (MODELS_DIR / 'best_model_config.json',
     MODELS_DIR / 'best_model.pt',
     'estandar (notebook 04)'),
]

MODEL = None; CLASS_NAMES = None; CFG = None; MODEL_LABEL = None
for cfg_path, ckpt_path, label in candidates:
    if (cfg_path.exists() and cfg_path.read_text().strip() not in ('', '{}')
            and ckpt_path.exists()):
        CFG = json.loads(cfg_path.read_text(encoding='utf-8'))
        MODEL = PretrainedModel(backbone=CFG['backbone'],
                                num_classes=CFG['num_classes'],
                                strategy=CFG['strategy'])
        MODEL.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        MODEL.eval().to(DEVICE)
        CLASS_NAMES = CFG['class_names']
        MODEL_LABEL = label
        print(f'Modelo cargado: {CFG["backbone"]} / {CFG["strategy"]} '
              f'({CFG["num_classes"]} clases) — variante {label}')
        if 'test_accuracy' in CFG:
            print(f'  Test accuracy en dataset: {CFG["test_accuracy"]:.4f}')
        break

if MODEL is None:
    print('ATENCION: no hay modelo entrenado. Ejecuta primero notebooks/04_transfer_learning.ipynb '
          'o scripts/retrain_handwritten.py')

C:\Users\motam\Escritorio\Máster Big Data 2025-2026\2º Cuatrimestre\Datos no estructurados\TRABAJO FINA´L\chemistry-recognizer\src\augmentation.py:105: UserWarning: Argument(s) 'fill_value' are not valid for transform CoarseDropout
  return A.CoarseDropout(num_holes_range=(2, 8),


Modelo cargado: resnet18 / finetune (196 clases) — variante handwritten-aug (robusto a dibujos a mano)
  Test accuracy en dataset: 0.9890


## A. Configuración del ejercicio

El panel deja al alumno acotar qué tipo de compuestos quiere practicar. Las subcategorías se actualizan dinámicamente cuando cambia la categoría (sólo se muestran las relevantes), y la lista de notaciones recoge las dos formas habituales en bachillerato: tradicional ("ácido sulfúrico") e IUPAC ("tetraoxosulfato(VI) de dihidrógeno"). Pulsar "Iniciar ejercicios" prepara el pool de compuestos filtrado.

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

category = widgets.RadioButtons(
    options=[('Inorgánica', 'inorganica'), ('Orgánica', 'organica'), ('Ambas', None)],
    description='Categoría:', value=None)
subcats = widgets.SelectMultiple(options=[], description='Subcat.:', rows=6)
difficulty = widgets.RadioButtons(
    options=[('Básico', 'basico'), ('Intermedio', 'intermedio'),
             ('Avanzado', 'avanzado'), ('Todos', None)],
    description='Dificultad:', value=None)
notation = widgets.RadioButtons(
    options=['Tradicional', 'IUPAC', 'Ambas'], description='Notación:', value='Ambas')
btn_start = widgets.Button(description='Iniciar ejercicios', button_style='success')
out_cfg = widgets.Output()

def refresh_subs(*_):
    opts = []
    cats = [category.value] if category.value else ['inorganica', 'organica']
    for c in cats:
        for sub, sub_data in TAXONOMY[c]['subcategories'].items():
            opts.append((sub_data['display'], sub))
    subcats.options = opts

category.observe(refresh_subs, names='value'); refresh_subs()
display(widgets.VBox([category, subcats, difficulty, notation, btn_start, out_cfg]))

## B–D. Tarjeta de ejercicio, canvas e inferencia

El bloque siguiente engloba los tres componentes que el alumno usa en cada ronda: la tarjeta con el compuesto a dibujar, el canvas de `ipycanvas` (con `FileUpload` como alternativa si la extensión no se ha instalado) y el botón "Comprobar" que dispara la inferencia. Para mantener la latencia por debajo del segundo se usa `VAL_TRANSFORM` de `src/augmentation.py` —el mismo que aplicamos en validación y test— y se hace forward en GPU si está disponible. El feedback visual incluye un top-5 con las probabilidades de las clases más confiables, lo que permite al alumno entender por qué se ha equivocado el modelo (o él) en lugar de mostrar solo "acierto/fallo".

In [4]:
import io
from PIL import Image
try:
    from ipycanvas import Canvas, hold_canvas
    HAS_CANVAS = True
except ImportError:
    HAS_CANVAS = False
    print('ipycanvas no disponible — se usará carga de fichero en su lugar.')

# Render de referencia (reusamos las funciones del generador del dataset)
from data.generate_dataset import render_base

state = {'pool': [], 'current': None, 'idx': 0,
         'score': {'total': 0, 'correct': 0, 'by_sub': {}}}


def preprocess_drawing(pil: Image.Image) -> Image.Image:
    """Acerca el dibujo del alumno al estilo de las imagenes de entrenamiento.

    Pasos:
      1. Pasamos a escala de grises.
      2. Binarizamos: cualquier pixel < 200 cuenta como trazo.
      3. Localizamos el bounding box de los pixeles negros y recortamos con padding.
      4. Centramos sobre un lienzo cuadrado blanco preservando la proporcion.
      5. Resize a 224x224 con LANCZOS y devolvemos en RGB.

    Si el lienzo esta en blanco devolvemos la imagen original.
    """
    g = np.array(pil.convert('L'))
    mask = g < 200
    if not mask.any():
        return pil.convert('RGB')

    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    pad = 20
    r0 = max(0, rmin - pad); r1 = min(g.shape[0], rmax + pad + 1)
    c0 = max(0, cmin - pad); c1 = min(g.shape[1], cmax + pad + 1)
    crop = g[r0:r1, c0:c1]

    # Binarizar limpio (sin anti-aliasing que confunda)
    crop = np.where(crop < 200, 0, 255).astype(np.uint8)

    # Centrar en cuadrado blanco
    h, w = crop.shape
    s = max(h, w)
    square = np.full((s, s), 255, dtype=np.uint8)
    y_off = (s - h) // 2; x_off = (s - w) // 2
    square[y_off:y_off + h, x_off:x_off + w] = crop

    out = Image.fromarray(square).resize((224, 224), Image.LANCZOS)
    return out.convert('RGB')


def predict_pil(img: Image.Image):
    if MODEL is None:
        return None, None
    arr = np.array(img.convert('RGB'))
    x = VAL_TRANSFORM(image=arr)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = MODEL(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy().squeeze()
    top5 = np.argsort(probs)[::-1][:5]
    return [(CLASS_NAMES[i], float(probs[i])) for i in top5], CLASS_NAMES[top5[0]]


out_exercise = widgets.Output()
out_feedback = widgets.Output()
out_preview = widgets.Output()  # vista previa del dibujo procesado tras Comprobar

btn_prev = widgets.Button(description='Anterior')
btn_next = widgets.Button(description='Siguiente')
btn_rand = widgets.Button(description='Aleatorio')
btn_check = widgets.Button(description='Comprobar', button_style='primary')
btn_clear = widgets.Button(description='Borrar')
btn_hint = widgets.ToggleButton(description='Ver pista')
pen_width = widgets.IntSlider(value=4, min=1, max=20, step=1, description='Grosor:')
eraser = widgets.ToggleButton(value=False, description='Goma', icon='eraser')

# Widget para mostrar el render de referencia
ref_image_widget = widgets.Image(format='png', width=200, height=200)
ref_caption = widgets.HTML('<small><i>Render de referencia (RDKit)</i></small>')


def update_ref_image():
    """Pinta en ref_image_widget el render canonico del compuesto actual."""
    c = state['current']
    if c is None:
        return
    try:
        ref_pil, _ = render_base(c)
        bio = io.BytesIO()
        ref_pil.resize((200, 200), Image.LANCZOS).save(bio, format='PNG')
        ref_image_widget.value = bio.getvalue()
    except Exception as e:
        print(f'No se pudo renderizar referencia: {e}')


if HAS_CANVAS:
    canvas = Canvas(width=400, height=400, sync_image_data=True)
    canvas.fill_style = 'white'; canvas.fill_rect(0, 0, 400, 400)
    canvas.stroke_style = 'black'; canvas.line_width = pen_width.value
    canvas.line_cap = 'round'; canvas.line_join = 'round'

    drawing = {'on': False, 'last': None}

    def _on_down(x, y):
        drawing['on'] = True
        drawing['last'] = (x, y)
        r = max(1, canvas.line_width // 2)
        canvas.fill_style = 'white' if eraser.value else 'black'
        canvas.fill_circle(x, y, r)

    def _on_move(x, y):
        if not drawing['on'] or drawing['last'] is None:
            return
        canvas.stroke_style = 'white' if eraser.value else 'black'
        canvas.line_width = pen_width.value if not eraser.value else max(pen_width.value * 3, 12)
        with hold_canvas(canvas):
            canvas.stroke_line(drawing['last'][0], drawing['last'][1], x, y)
        drawing['last'] = (x, y)

    def _on_up(x, y):
        drawing['on'] = False
        drawing['last'] = None

    canvas.on_mouse_down(_on_down)
    canvas.on_mouse_move(_on_move)
    canvas.on_mouse_up(_on_up)
    canvas.on_mouse_out(_on_up)

    def _on_pen_change(change):
        canvas.line_width = change['new']
    pen_width.observe(_on_pen_change, names='value')
else:
    upload = widgets.FileUpload(accept='image/*', multiple=False, description='Sube dibujo')


def render_exercise():
    with out_exercise:
        clear_output()
        c = state['current']
        if c is None:
            print('Configura y pulsa "Iniciar ejercicios".'); return
        diff_emoji = {'basico': '🟢', 'intermedio': '🟡', 'avanzado': '🔴'}.get(c['difficulty'], '')
        msg = f"### Dibuja: **{c['name_display']}**  {diff_emoji}\n\n"
        if notation.value in ('IUPAC', 'Ambas'):
            msg += f"_IUPAC: {c['name_iupac']}_\n\n"
        if btn_hint.value:
            msg += f"Pista: `{c['formula_display']}`"
        display(widgets.HTML('<style>.md{font-family:sans-serif}</style>'))
        display(widgets.HTML('<div class="md">' + msg.replace('\n', '<br>') + '</div>'))
    update_ref_image()


def pick_pool(_):
    pool = get_compounds(category=category.value,
                         subcategories=list(subcats.value) or None,
                         difficulty=difficulty.value)
    if not pool:
        with out_cfg: print('No hay compuestos con esos filtros.'); return
    random.shuffle(pool)
    state['pool'] = pool; state['idx'] = 0; state['current'] = pool[0]
    render_exercise()


def next_ex(_):
    if not state['pool']: return
    state['idx'] = (state['idx'] + 1) % len(state['pool'])
    state['current'] = state['pool'][state['idx']]; render_exercise()


def prev_ex(_):
    if not state['pool']: return
    state['idx'] = (state['idx'] - 1) % len(state['pool'])
    state['current'] = state['pool'][state['idx']]; render_exercise()


def rand_ex(_):
    if not state['pool']: return
    state['current'] = random.choice(state['pool']); render_exercise()


def clear_canvas(_):
    if HAS_CANVAS:
        canvas.clear(); canvas.fill_style = 'white'; canvas.fill_rect(0, 0, 400, 400)


def check(_):
    c = state['current']
    if c is None: return
    if HAS_CANVAS:
        img_data = canvas.get_image_data()
        raw_pil = Image.fromarray(img_data[:, :, :3].astype(np.uint8))
    else:
        if not upload.value:
            with out_feedback: clear_output(); print('Sube primero una imagen.'); return
        f = list(upload.value.values())[0]
        raw_pil = Image.open(io.BytesIO(f['content']))

    processed = preprocess_drawing(raw_pil)
    top5, pred = predict_pil(processed)

    if pred is None:
        with out_feedback: clear_output(); print('Modelo no cargado.'); return
    ok = (pred == c['id'])
    state['score']['total'] += 1
    state['score']['correct'] += int(ok)
    sub = c['subcategory']
    state['score']['by_sub'].setdefault(sub, {'total': 0, 'correct': 0})
    state['score']['by_sub'][sub]['total'] += 1
    state['score']['by_sub'][sub]['correct'] += int(ok)
    color = 'green' if ok else 'red'
    icon = '✅' if ok else '❌'

    with out_preview:
        clear_output()
        # Mostramos el dibujo ya preprocesado para que se vea lo que ve el modelo
        bio = io.BytesIO(); processed.save(bio, format='PNG')
        display(widgets.HTML('<small><i>Tu dibujo tras preprocesar (lo que ve el modelo):</i></small>'))
        display(widgets.Image(value=bio.getvalue(), format='png', width=180, height=180))

    with out_feedback:
        clear_output()
        display(HTML(f'<div style="padding:10px;background:#f3f3f3;border-left:6px solid {color}">'
                     f'{icon} Predicción: <b>{pred}</b><br>Esperado: <b>{c["id"]}</b> ({c["formula_display"]})</div>'))
        import matplotlib.pyplot as plt
        names = [n for n, _ in top5]; probs = [p for _, p in top5]
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.barh(names[::-1], probs[::-1], color='steelblue'); ax.set_xlim(0, 1)
        ax.set_title('Top-5 confianza'); plt.tight_layout(); plt.show()


btn_start.on_click(pick_pool)
btn_next.on_click(next_ex); btn_prev.on_click(prev_ex); btn_rand.on_click(rand_ex)
btn_clear.on_click(clear_canvas); btn_check.on_click(check)
btn_hint.observe(lambda *_: render_exercise(), names='value')

controls_row1 = widgets.HBox([btn_prev, btn_next, btn_rand, btn_hint])
controls_row2 = widgets.HBox([btn_check, btn_clear, pen_width, eraser])

ref_panel = widgets.VBox([ref_caption, ref_image_widget, out_preview])
if HAS_CANVAS:
    drawing_area = widgets.HBox([canvas, ref_panel])
else:
    drawing_area = widgets.HBox([upload, ref_panel])

main_panel = [out_exercise, controls_row1, drawing_area, controls_row2, out_feedback]
display(widgets.VBox(main_panel))

## E. Marcador

Botón aparte para que el alumno pueda revisar su progreso cuando quiera. El marcador se mantiene a nivel global (intentos / aciertos / accuracy) y desglosado por subcategoría —útil para detectar dónde flojea el alumno: si lleva 8/10 en alcanos pero 1/9 en oxoácidos, sabe que es ahí donde tiene que dedicar tiempo. El mensaje final es deliberadamente sencillo: animar si las cosas van bien y sugerir repasar si no.

In [5]:
btn_score = widgets.Button(description='Mostrar marcador', button_style='info')
out_score = widgets.Output()
def show_score(_):
    s = state['score']
    with out_score:
        clear_output()
        if s['total'] == 0:
            print('Aún no has intentado ningún ejercicio.'); return
        acc = s['correct'] / s['total']
        print(f'Intentos: {s["total"]}  Aciertos: {s["correct"]}  Accuracy: {acc:.1%}')
        if s['by_sub']:
            print('\nPor subcategoría:')
            for sub, d in s['by_sub'].items():
                a = d['correct'] / max(d['total'], 1)
                print(f'  {sub:25s} {d["correct"]:3d}/{d["total"]:3d}  ({a:.1%})')
        if acc >= 0.8: print('\n¡Excelente! Dominas el material.')
        elif acc >= 0.5: print('\nVas por buen camino, sigue practicando.')
        else: print('\nRepasa las nomenclaturas y vuelve a intentarlo.')
btn_score.on_click(show_score)
display(widgets.VBox([btn_score, out_score]))

## F. Modo "Ver al modelo trabajar" sobre el dataset

La sección anterior pide al alumno que **dibuje** un compuesto y el modelo intenta reconocerlo. Esa interacción es la más vistosa pero, como advertimos en la sección de limitaciones, falla con frecuencia por el cambio de dominio (RDKit vs mano humana).

El bloque siguiente ofrece la interacción complementaria: **el alumno ve una imagen aleatoria del split de test del dataset, y el modelo la clasifica delante de sus ojos**. Aquí el modelo está trabajando sobre el dominio para el que fue entrenado, así que la accuracy es la real de test (~99,5%). Sirve para mostrar el modelo bien empleado y como contraste con la sección anterior.

Cada vez que pulsamos *"Otra imagen"* tomamos un compuesto al azar del split de test, lo pasamos por el modelo, y mostramos su top-5 + la imagen + el nombre real para que se pueda juzgar a simple vista si la predicción es correcta.

In [6]:
import pandas as pd
from PIL import Image as PILImage
import matplotlib.pyplot as plt

# Cargamos el split de test
df_test = pd.read_csv(ROOT / 'data' / 'metadata.csv')
df_test = df_test[df_test['split'] == 'test'].reset_index(drop=True)

quiz_state = {'current_row': None}
out_quiz_image = widgets.Output()
out_quiz_result = widgets.Output()

btn_quiz_pick = widgets.Button(description='Otra imagen', button_style='success')
btn_quiz_reveal = widgets.Button(description='Revelar', button_style='warning')


def predict_from_path(fp):
    img = PILImage.open(ROOT / fp).convert('RGB')
    arr = np.array(img)
    x = VAL_TRANSFORM(image=arr)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = MODEL(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy().squeeze()
    top5_idx = np.argsort(probs)[::-1][:5]
    top5 = [(CLASS_NAMES[i], float(probs[i])) for i in top5_idx]
    return img, top5, CLASS_NAMES[top5_idx[0]]


def quiz_pick(_):
    row = df_test.sample(1).iloc[0]
    quiz_state['current_row'] = row
    img, top5, pred = predict_from_path(row['filepath'])

    with out_quiz_image:
        clear_output()
        display(widgets.HTML(f'<h3>¿Qué compuesto es?</h3>'))
        bio = io.BytesIO(); img.save(bio, format='PNG')
        display(widgets.Image(value=bio.getvalue(), format='png', width=220, height=220))
        display(widgets.HTML('<small><i>Imagen del split de test (el modelo no la vio en train ni val).</i></small>'))

    with out_quiz_result:
        clear_output()
        ok = (pred == row['compound_id'])
        color = 'green' if ok else 'red'
        icon = '✅' if ok else '❌'
        display(HTML(
            f'<div style="padding:10px;background:#f3f3f3;border-left:6px solid {color}">'
            f'{icon} Predicción del modelo: <b>{pred}</b><br>'
            f'Compuesto real: <b>{row["compound_id"]}</b> ({row["formula_display"]}) — '
            f'<i>{row["name_display"]}</i>'
            f'</div>'))
        names = [n for n, _ in top5]; probs = [p for _, p in top5]
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.barh(names[::-1], probs[::-1], color='steelblue'); ax.set_xlim(0, 1)
        ax.set_title('Top-5 confianza del modelo')
        plt.tight_layout(); plt.show()


btn_quiz_pick.on_click(quiz_pick)
display(widgets.VBox([widgets.HBox([btn_quiz_pick]),
                       widgets.HBox([out_quiz_image, out_quiz_result])]))